# Proton therapy: the Bragg peak

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/proton_bragg_peak)

The Bethe–Bloch stopping power $-dE/dz \propto 1/\beta^2$ is integrated with RK4 for a whole batch of proton energies at once. The dose piles up just before each proton stops (the Bragg peak), and a weighted sum of energy layers gives a flat dose over a tumour with almost nothing behind it.

**Vector form**

$$-\frac{dE}{dz} = K\rho\,\frac{Z}{A}\,\frac{1}{\beta^2}\left[\tfrac12\ln\frac{2m_ec^2\beta^2\gamma^2T_{\max}}{I^2} - \beta^2\right] \qquad D^{\rm SOBP}(z) = \sum_k w_k\,D_k(z)$$

**Index (tensor) form**

$$\frac{dE_b}{dz} = -S(E_b) \qquad D_{bz} = G_{bzk}\,d_{bk} \qquad D^{\rm SOBP}_z = D_{kz}\,w_k,\; w_k \ge 0$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $E_b$ | `E[:, b]` | (Z+1, B) | kinetic energy of beam $b$ at every depth step (RK4, all beams together) |
| $S(E_b)$ | `stopping_power(E, I)` | (B,) | Bethe–Bloch $-dE/dz$ in MeV/cm, elementwise over the batch |
| $d_{bk}$ | `-diff(E, dim=0) / dz` | (B, Z) | pristine dose: energy lost per cm in depth bin $k$ |
| $G_{bzk}$ | `exp(-0.5 * (diff / sigma) ** 2)` | (B, Z, Z) | Gaussian range straggling + energy spread, rows normalised |
| $G_{bzk}\,d_{bk}$ | `einsum('bzk,bk->bz', G, d)` | (B, Z) | repeated $k$ ⇒ summed: blurred Bragg curve |
| $D_{kz}\,w_k$ | `einsum('kz,k->z', layers, w)` | (Z,) | repeated $k$ ⇒ summed: spread-out Bragg peak from $K$ energy layers |
| $w_k$ | `sobp_weights(layers, z, z1, z2)` | (K,) | non-negative least squares (projected gradient) for a flat target dose |

Constants for water: $K = 0.307$ MeV cm²/mol, $Z/A = 0.555$, $I \approx 75$ eV, $\rho = 1$ g/cm³; shell and density corrections are ignored. The integrated range (150 MeV → about 15.8 cm) agrees with NIST PSTAR to about 1 %, and fits the Bragg–Kleeman rule $R = \alpha E^p$ with $p \approx 1.75$. Range straggling ($\sigma \approx 1.2\,\%$ of $R$) and the beam energy spread round the peak; nuclear reactions are modelled only as a 1 %/cm loss of primaries, so the peak-to-entrance ratio is higher than in a real beam. A megavoltage X-ray beam (build-up, then $e^{-\mu z}$) normalised to the same tumour dose deposits more at the entrance and keeps going beyond the tumour; protons stop. All numbers are approximate and for education only, not for clinical use or treatment planning.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/proton_bragg_peak/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

DZ = 0.04  # depth step (cm)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    I_mev = params["i_ev"] * 1e-6
    spread = params["energy_spread"] / 100.0
    z1, z2 = params["target_start"], max(params["target_end"], params["target_start"] + 1.0)

    # 1) A batch of pencil beams: slow down, find the range, compute the depth-dose curves.
    E0 = tensors.beam_energies(params["e_min"], params["e_max"], params["n_energies"], dev)  # (B,)
    z_max = max(1.15 * float(equation.bragg_kleeman(E0.max(), 0.0022, 1.77)) + 2.0, z2 + 8.0)
    z, E = solve.slow_down(E0, I_mev, z_max, DZ)  # (Z+1,), (Z+1, B)
    ranges = solve.csda_range(z, E)
    z_mid, dose = solve.depth_dose(z, E, ranges, spread)  # (B, Z)
    times = solve.flight_times(z, E)
    alpha, p = solve.fit_bragg_kleeman(E0, ranges) if len(E0) > 1 else (0.0022, 1.77)

    # 2) Spread-out Bragg peak: energy layers whose ranges cover the target, non-negative weights.
    E_layers = tensors.layer_energies(tensors.layer_ranges(z1, z2, params["n_layers"], dev), alpha, p)
    zl, El = solve.slow_down(E_layers, I_mev, z_max, DZ)
    _, layer_dose = solve.depth_dose(zl, El, solve.csda_range(zl, El), spread)  # (K, Z)
    w = solve.sobp_weights(layer_dose, z_mid, z1, z2)
    plateau = equation.sobp(layer_dose, w)
    xray = equation.xray_depth_dose(z_mid)
    centre = (z_mid - 0.5 * (z1 + z2)).abs().argmin()
    xray = xray / xray[centre]  # same dose as the protons in the middle of the tumour

    print(f"stopping power at 150 MeV: {float(equation.stopping_power(E0.new_tensor(150.0), I_mev)):.2f} MeV/cm")
    for e, r, zp in zip(E0.tolist(), ranges.tolist(), z_mid[dose.argmax(1)].tolist()):
        print(f"  E0 = {e:6.1f} MeV  range {r:6.2f} cm  dose peak at {zp:6.2f} cm")
    print(f"Bragg–Kleeman fit: R = {alpha:.5f} E^{p:.3f}")
    beyond = z_mid > z2 + 1.0
    print(
        f"SOBP over {z1:.1f}–{z2:.1f} cm with {len(w)} layers: "
        f"flatness {100 * solve.flatness(plateau, z_mid, z1, z2):.2f} %"
    )
    near = (z_mid - 1.0).abs().argmin()  # 1 cm deep, past the photon build-up
    print(f"dose at 1 cm depth: protons {float(plateau[near]):.2f}, X-rays {float(xray[near]):.2f}  (target = 1)")
    print(
        f"mean dose beyond the target: protons {float(plateau[beyond].mean()):.3f}, "
        f"X-rays {float(xray[beyond].mean()):.3f}"
    )

    layers = layer_dose * w[:, None]  # (K, Z) each layer's weighted contribution
    return plot.animate(
        z,
        times,
        E0,
        ranges,
        z_mid,
        dose,
        (alpha, p),
        tensors.reference_ranges(dev),
        layers,
        xray,
        (z1, z2),
        out_dir / "proton_bragg_peak.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/proton_bragg_peak/equation.py
r"""Physics → tensor translation.

A proton loses energy to atomic electrons at the rate given by the Bethe–Bloch formula
(no shell or density corrections), for water of density ρ:

    -\frac{dE}{dz} = K \rho \frac{Z}{A} \frac{1}{\beta^2}
                     \left[\tfrac12 \ln\frac{2 m_e c^2 \beta^2\gamma^2 T_{max}}{I^2} - \beta^2\right]

It grows like 1/β² as the proton slows down, so most energy is dumped just before it stops: the Bragg peak.
For a batch b of initial energies the ODE is integrated together, E_b(z) of shape (B,).

The depth-dose of pencil b, blurred by range straggling (Gaussian kernel G_{bzk}), and a spread-out
Bragg peak as a weighted sum of energy layers k:

    D_{bz} = G_{bzk}\, d_{bk}                (k = depth bin, summed)
    D^{SOBP}_z = D_{kz}\, w_k                (k = energy layer, summed)
"""

import torch

M_E = 0.51099895  # electron rest energy (MeV)
M_P = 938.272  # proton rest energy (MeV)
K = 0.307075  # 4π N_A r_e² m_e c² (MeV cm²/mol)
Z_OVER_A = 0.5551  # water: 10 electrons per 18.015 g/mol
RHO = 1.0  # g/cm³


def kinematics(E: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """(β², β²γ², γ) for a proton of kinetic energy E (MeV); any shape."""
    gamma = 1.0 + E / M_P
    bg2 = gamma**2 - 1.0
    return bg2 / gamma**2, bg2, gamma


def stopping_power(E: torch.Tensor, I_mev: float) -> torch.Tensor:
    r"""-dE/dz in MeV/cm from the Bethe–Bloch formula (no shell or density correction)."""
    beta2, bg2, gamma = kinematics(E)
    t_max = 2 * M_E * bg2 / (1 + 2 * gamma * M_E / M_P + (M_E / M_P) ** 2)  # max energy to one electron
    log_term = 0.5 * torch.log(2 * M_E * bg2 * t_max / I_mev**2)
    return K * RHO * Z_OVER_A / beta2 * (log_term - beta2)


def rhs(E: torch.Tensor, I_mev: float, e_cut: float) -> torch.Tensor:
    """dE/dz for the batch (B,): the proton stops (rate 0) once E falls below e_cut, where Bethe fails."""
    moving = E > e_cut
    return torch.where(moving, -stopping_power(E.clamp_min(e_cut), I_mev), torch.zeros_like(E))


def bragg_kleeman(E: torch.Tensor, alpha: float, p: float) -> torch.Tensor:
    r"""Empirical range–energy rule R = \alpha E^p (cm, E in MeV)."""
    return alpha * E**p


def straggling_width(R: torch.Tensor) -> torch.Tensor:
    """Range straggling σ ≈ 0.012 R^0.951 (cm) in water — about 1.2 % of the range."""
    return 0.012 * R.clamp_min(1e-3) ** 0.951


def primary_fluence(z: torch.Tensor, kappa: float = 0.011) -> torch.Tensor:
    """Fraction of protons not yet removed by nuclear reactions, ≈ 1 % per cm (crude)."""
    return torch.exp(-kappa * z)


def straggle(d: torch.Tensor, z: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    r"""D_{bz} = G_{bzk} d_{bk}: blur each pristine curve with a Gaussian of width σ_b (cm).

    d: (B, Z) pristine dose, z: (Z,), sigma: (B,). Each row of G sums to 1 (a local weighted average),
    so a flat curve stays flat right up to the surface.
    """
    diff = z[None, :, None] - z[None, None, :]  # (1, Z, Z)  z_z − z_k
    G = torch.exp(-0.5 * (diff / sigma[:, None, None]) ** 2)  # (B, Z, Z)
    G = G / G.sum(dim=2, keepdim=True)
    return torch.einsum("bzk,bk->bz", G, d)


def sobp(layers: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""D^{SOBP}_z = D_{kz} w_k.   layers: (K, Z), w: (K,) → (Z,)"""
    return torch.einsum("kz,k->z", layers, w)


def xray_depth_dose(z: torch.Tensor, mu: float = 0.049, buildup: float = 2.5) -> torch.Tensor:
    r"""Megavoltage photon beam: D(z) ∝ (1 - e^{-b z})\, e^{-\mu z} (electron build-up, then exponential)."""
    return (1 - torch.exp(-buildup * z)) * torch.exp(-mu * z)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/proton_bragg_peak/tensors.py
"""Build the concrete tensors: the batch of beam energies and reference ranges."""

import torch

# NIST PSTAR CSDA ranges of protons in water (g/cm² = cm at ρ = 1), rounded.
PSTAR = {70.0: 4.08, 100.0: 7.72, 150.0: 15.77, 200.0: 25.96, 250.0: 37.94}


def beam_energies(e_min: float, e_max: float, n: int, device: torch.device) -> torch.Tensor:
    """E0_b evenly spread between e_min and e_max (MeV): shape (B,)."""
    if n == 1:
        return torch.tensor([0.5 * (e_min + e_max)], device=device)
    return torch.linspace(e_min, e_max, n, device=device)


def reference_ranges(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """PSTAR (energy (R,), CSDA range (R,)) for comparison."""
    return torch.tensor(list(PSTAR), device=device), torch.tensor(list(PSTAR.values()), device=device)


def layer_ranges(z_start: float, z_end: float, n_layers: int, device: torch.device) -> torch.Tensor:
    """Ranges of the SOBP energy layers, spread a little beyond the target on both sides: (K,)."""
    return torch.linspace(z_start - 0.5, z_end + 0.5, n_layers, device=device)


def layer_energies(ranges: torch.Tensor, alpha: float, p: float) -> torch.Tensor:
    """Invert Bragg–Kleeman R = α E^p: the energy that stops at each wanted depth: (K,)."""
    return (ranges / alpha) ** (1.0 / p)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/proton_bragg_peak/solve.py
"""Slow the protons down (RK4 in depth), turn energy loss into dose, fit the range law and build an SOBP."""

import torch

from problems._common.integrators import integrate
from problems._common.progress import steps

from . import equation

E_CUT = 0.1  # MeV: below this the proton is considered stopped (residual range ≈ 1 µm)
C_CM_PER_NS = 29.9792458


def slow_down(E0: torch.Tensor, I_mev: float, z_max: float, dz: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Integrate dE/dz = −S(E) for the whole batch. Returns (z (Z+1,), E (Z+1, B))."""
    n = int(round(z_max / dz))
    E = integrate(lambda e, z: equation.rhs(e, I_mev, E_CUT), E0, dz, n)  # (Z+1, B)
    z = torch.arange(n + 1, device=E0.device) * dz
    return z, E.clamp_min(0.0)


def csda_range(z: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    """Depth where each proton stops (E ≤ E_CUT), linearly interpolated between steps: (B,)."""
    stopped = E <= E_CUT  # (Z+1, B)
    k = torch.where(stopped.any(0), stopped.int().argmax(0), len(z) - 1).clamp_min(1)  # first stopped step
    b = torch.arange(E.shape[1], device=E.device)
    e_before, e_after = E[k - 1, b], E[k, b]
    frac = ((e_before - E_CUT) / (e_before - e_after).clamp_min(1e-9)).clamp(0, 1)
    return z[k - 1] + frac * (z[1] - z[0])


def depth_dose(
    z: torch.Tensor, E: torch.Tensor, ranges: torch.Tensor, energy_spread: float = 0.0, p: float = 1.75
) -> tuple[torch.Tensor, torch.Tensor]:
    """Dose per proton (MeV/cm) at the bin centres for each beam: (z_mid (Z,), D (B, Z)).

    The energy lost in each depth step, −ΔE/Δz, is the pristine Bragg curve; we then weight it by the
    surviving primary fluence and blur it with range straggling plus the beam's energy spread
    (a relative spread σ_E/E widens the range by σ_R ≈ p (σ_E/E) R, since R ∝ E^p).
    """
    dz = z[1] - z[0]
    z_mid = 0.5 * (z[1:] + z[:-1])
    pristine = (-torch.diff(E, dim=0) / dz).T  # (B, Z)
    pristine = pristine * equation.primary_fluence(z_mid)[None, :]
    sigma = torch.sqrt(equation.straggling_width(ranges) ** 2 + (p * energy_spread * ranges) ** 2)
    return z_mid, equation.straggle(pristine, z_mid, sigma)


def fit_bragg_kleeman(E0: torch.Tensor, R: torch.Tensor) -> tuple[float, float]:
    """Least squares on log R = log α + p log E. Returns (α, p)."""
    A = torch.stack([torch.ones_like(E0), torch.log(E0)], dim=1)  # (B, 2)
    coef = torch.linalg.lstsq(A, torch.log(R)[:, None]).solution[:, 0]
    return float(torch.exp(coef[0])), float(coef[1])


def sobp_weights(layers: torch.Tensor, z: torch.Tensor, z1: float, z2: float, iters: int = 3000) -> torch.Tensor:
    """Non-negative weights w_k making D_{kz} w_k ≈ 1 on the target [z1, z2] (projected gradient NNLS)."""
    target = (z >= z1) & (z <= z2)
    A = layers[:, target].T  # (Z_T, K)
    ones = torch.ones(A.shape[0], device=A.device)
    w = torch.linalg.lstsq(A, ones[:, None]).solution[:, 0].clamp_min(0.0)  # unconstrained start, clipped
    step = 1.0 / torch.linalg.matrix_norm(A, ord=2) ** 2  # 1 / Lipschitz constant of the gradient
    for _ in steps(iters, "NNLS iterations"):
        w = (w - step * A.T @ (A @ w - ones)).clamp_min(0.0)
    return w


def flight_times(z: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    """Time (ns) for each proton to reach depth z: t = ∫ dz / v.  (Z+1, B); stays constant once stopped."""
    beta = torch.sqrt(equation.kinematics(E.clamp_min(E_CUT))[0])
    moving = (E > E_CUT).float()
    dt = (z[1] - z[0]) / (beta * C_CM_PER_NS) * moving
    return torch.cat([torch.zeros_like(dt[:1]), torch.cumsum(dt[:-1], dim=0)])


def flatness(dose: torch.Tensor, z: torch.Tensor, z1: float, z2: float) -> float:
    """(max − min) / mean of the dose over the target."""
    d = dose[(z >= z1) & (z <= z2)]
    return float((d.max() - d.min()) / d.mean())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/proton_bragg_peak/plot.py
"""Animate: protons slow down in water while their dose builds up, then the SOBP is assembled layer by layer."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

FLIGHT, BUILD = 60, 40  # logical steps of the two phases


def animate(
    z: torch.Tensor,  # (Z+1,) depth grid
    times: torch.Tensor,  # (Z+1, B) ns
    E0: torch.Tensor,  # (B,)
    ranges: torch.Tensor,  # (B,)
    z_mid: torch.Tensor,  # (Z,)
    dose: torch.Tensor,  # (B, Z)
    fit: tuple[float, float],
    reference: tuple[torch.Tensor, torch.Tensor],
    layers: torch.Tensor,  # (K, Z) weighted layer doses
    xray: torch.Tensor,  # (Z,)
    target: tuple[float, float],
    path: Path,
) -> Path:
    z, times, E0, ranges = z.cpu().numpy(), times.cpu().numpy(), E0.cpu().numpy(), ranges.cpu().numpy()
    z_mid, dose, layers, xray = z_mid.cpu().numpy(), dose.cpu().numpy(), layers.cpu().numpy(), xray.cpu().numpy()
    e_ref, r_ref = (r.cpu().numpy() for r in reference)
    alpha, p = fit
    z1, z2 = target
    n_b, n_k = len(E0), len(layers)
    colors = [plt.get_cmap("viridis")(0.1 + 0.8 * b / max(1, n_b - 1)) for b in range(n_b)]
    dose_norm = dose / dose.max()
    t_end = times[-1].max() * 1.02

    fig, axes = plt.subplots(2, 2, figsize=(9, 5.4))
    (ax_tank, ax_range), (ax_dose, ax_sobp) = axes
    z_max = z[-1]

    ax_tank.axvspan(0, z_max, color="#dbeafe")
    ax_tank.set_xlim(-1, z_max)
    ax_tank.set_ylim(-0.7, n_b - 0.3)
    ax_tank.set_yticks(range(n_b), [f"{e:.0f}" for e in E0])
    ax_tank.set_ylabel("$E_0$ (MeV)")
    ax_tank.set_xlabel("depth in water (cm)")
    trails = [ax_tank.plot([], [], color=colors[b], lw=2)[0] for b in range(n_b)]
    heads = [ax_tank.plot([], [], "o", color=colors[b], ms=6, mec="#1e40af")[0] for b in range(n_b)]
    clock = ax_tank.set_title("protons in water")

    e_line = np.linspace(min(E0.min(), 50), max(E0.max(), 250), 100)
    ax_range.plot(e_line, alpha * e_line**p, color="#9ca3af", lw=1.5, label=f"$R = {alpha:.4f}\\,E^{{{p:.2f}}}$")
    ax_range.plot(e_ref, r_ref, "x", color="#dc2626", ms=7, label="PSTAR")
    range_dots = ax_range.scatter([], [], color=ACCENT[0], s=22, zorder=3, label="Bethe–Bloch")
    ax_range.set_xlabel("$E_0$ (MeV)")
    ax_range.set_ylabel("range (cm)")
    ax_range.legend(fontsize=8, loc="upper left")

    dose_lines = [ax_dose.plot([], [], color=colors[b], lw=1.4)[0] for b in range(n_b)]
    ax_dose.set_xlim(0, z_max)
    ax_dose.set_ylim(0, 1.08)
    ax_dose.set_xlabel("depth (cm)")
    ax_dose.set_ylabel("dose (rel.)")
    ax_dose.set_title("pristine Bragg peaks")

    ax_sobp.axvspan(z1, z2, color="#fde68a", alpha=0.6, lw=0)
    layer_lines = [ax_sobp.plot([], [], color=ACCENT[0], lw=0.7, alpha=0.6)[0] for _ in range(n_k)]
    (sum_line,) = ax_sobp.plot([], [], color=ACCENT[1], lw=2, label="protons (SOBP)")
    (x_line,) = ax_sobp.plot([], [], color="#374151", lw=1.4, ls="--", label="X-rays")
    ax_sobp.set_xlim(0, max(z2 + 6, 0.6 * z_max))
    ax_sobp.set_ylim(0, 1.9)
    ax_sobp.set_xlabel("depth (cm)")
    ax_sobp.set_ylabel("dose (rel.)")
    ax_sobp.set_title("spread-out Bragg peak")
    ax_sobp.legend(fontsize=8, loc="upper right")

    def update(step: int):
        T = t_end * min(1.0, step / (FLIGHT - 1))
        stopped = []
        for b in range(n_b):
            depth = min(np.interp(T, times[:, b], z), ranges[b])
            trails[b].set_data([0, depth], [b, b])
            heads[b].set_data([depth], [b])
            done = depth >= ranges[b] - 1e-6
            shown = np.ones_like(z_mid, bool) if done else z_mid <= depth  # full curve once stopped
            dose_lines[b].set_data(z_mid[shown], dose_norm[b, shown])
            if done:
                stopped.append(b)
        clock.set_text(f"protons in water   t = {T:.2f} ns")
        range_dots.set_offsets(np.column_stack([E0[stopped], ranges[stopped]]) if stopped else np.empty((0, 2)))

        n_shown = 0 if step < FLIGHT else min(n_k, 1 + (step - FLIGHT) * n_k // (BUILD - 8))
        order = np.arange(n_k)[::-1]  # deepest layer first
        for i, k in enumerate(order):
            layer_lines[k].set_data((z_mid, layers[k]) if i < n_shown else ([], []))
        partial = layers[order[:n_shown]].sum(0) if n_shown else None
        sum_line.set_data((z_mid, partial) if n_shown else ([], []))
        x_line.set_data((z_mid, xray) if n_shown == n_k else ([], []))

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(FLIGHT + BUILD, 80), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'e_min': 70,
    'e_max': 230,
    'n_energies': 7,
    'target_start': 10.0,
    'target_end': 15.0,
    'n_layers': 16,
    'energy_spread': 1.0,
    'i_ev': 75
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.proton_bragg_peak")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.proton_bragg_peak.main")
problem_dir = Path("problems/medical/proton_bragg_peak")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
